# Clasificación de riesgo de plagas

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

Carga de datos

In [3]:
df_ag = pd.read_csv('datos/produccion_agricola.csv')
df_clima = pd.read_csv('datos/condiciones_climaticas_limpios.csv')

In [9]:
clima_pais = df_clima.groupby(['anio', 'pais']).mean(numeric_only=True).reset_index()
clima_pais

,anio,pais,dias_con_heladas,humedad_relativa_promedio,indice_aridez,meses_estres_hidrico,precipitacion_total,temperatura_maxima,temperatura_minima,temperatura_promedio
0,2011,Alemania,72.00,76.00,24.860000,0.000000,886.000000,18.200000,1.800000,9.900000
1,2011,Argentina,51.00,69.00,14.753333,4.666667,795.666667,26.133333,4.733333,16.366667
2,2011,Australia,93.60,77.40,14.960000,0.800000,975.800000,31.360000,8.660000,21.000000
3,2011,Brasil,66.00,61.00,8.372000,8.800000,547.000000,31.100000,5.140000,18.620000
4,2011,China,40.00,54.00,13.350000,2.000000,817.000000,25.500000,6.600000,17.000000
...,...,...,...,...,...,...,...,...,...,...
95,2020,Estados Unidos,37.40,46.20,17.640000,3.800000,1036.600000,29.000000,5.600000,16.260000
96,2020,Francia,71.00,43.00,27.920000,4.000000,1387.000000,21.400000,0.000000,13.800000
97,2020,India,92.25,57.25,10.035000,5.250000,647.000000,31.175000,9.525000,17.600000
98,2020,Kenia,111.00,64.00,11.840000,0.000000,938.000000,35.700000,9.100000,22.000000


In [10]:
df = pd.merge(df_ag, clima_pais, on=['anio', 'pais'], how='inner')
df = df.dropna()

In [11]:
df

,pais,codigo_iso,region,cultivo,anio,superficie_hectareas,rendimiento_ton_ha,produccion_ton,fertilizantes_kg_ha,agua_riego_m3_ha,tendencia_5_anios,dias_con_heladas,humedad_relativa_promedio,indice_aridez,meses_estres_hidrico,precipitacion_total,temperatura_maxima,temperatura_minima,temperatura_promedio
0,Argentina,ARG,América del Sur,Soja,2011,4703643,2.22,10465461,105,4811,0.000536,51.0,69.0,14.753333,4.666667,795.666667,26.133333,4.733333,16.366667
1,Argentina,ARG,América del Sur,Maíz,2011,3371781,8.45,28485921,164,5557,0.030943,51.0,69.0,14.753333,4.666667,795.666667,26.133333,4.733333,16.366667
2,Argentina,ARG,América del Sur,Cebada,2011,1661577,2.77,4596917,105,6514,-0.039779,51.0,69.0,14.753333,4.666667,795.666667,26.133333,4.733333,16.366667
3,Argentina,ARG,América del Sur,Té,2011,188518,3.07,577886,236,8527,0.003623,51.0,69.0,14.753333,4.666667,795.666667,26.133333,4.733333,16.366667
4,Argentina,ARG,América del Sur,Algodón,2011,1638327,1.18,1929756,276,6925,0.007735,51.0,69.0,14.753333,4.666667,795.666667,26.133333,4.733333,16.366667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,Kenia,KEN,África,Arroz,2020,2415943,6.83,16508239,70,6500,0.023576,111.0,64.0,11.840000,0.000000,938.000000,35.700000,9.100000,22.000000
796,Kenia,KEN,África,Maíz,2020,5214983,7.47,38945261,208,1669,-0.013443,111.0,64.0,11.840000,0.000000,938.000000,35.700000,9.100000,22.000000
797,Kenia,KEN,África,Té,2020,262109,2.67,700065,85,6487,-0.026137,111.0,64.0,11.840000,0.000000,938.000000,35.700000,9.100000,22.000000
798,Kenia,KEN,África,Girasol,2020,3483346,3.64,12677903,229,4437,0.025317,111.0,64.0,11.840000,0.000000,938.000000,35.700000,9.100000,22.000000


Creación de variable sintética para predicción

In [ ]:
# Definimos "Alto Riesgo" (1) como años donde el rendimiento es bajo (percentil 30)
# Y hubo condiciones de estrés (aridez o meses de estrés hídrico altos)
threshold = df['rendimiento_ton_ha'].quantile(0.30)
stress = (df['meses_estres_hidrico'] > df['meses_estres_hidrico'].mean()) | (df['indice_aridez'] > df['indice_aridez'].mean())
df['alto_riesgo'] = ((df['rendimiento_ton_ha'] < threshold) & stress).astype(int)

In [22]:
df

,pais,codigo_iso,region,cultivo,anio,superficie_hectareas,rendimiento_ton_ha,produccion_ton,fertilizantes_kg_ha,agua_riego_m3_ha,tendencia_5_anios,dias_con_heladas,humedad_relativa_promedio,indice_aridez,meses_estres_hidrico,precipitacion_total,temperatura_maxima,temperatura_minima,temperatura_promedio,alto_riesgo
0,Argentina,ARG,América del Sur,Soja,2011,4703643,2.22,10465461,105,4811,0.000536,51.0,69.0,14.753333,4.666667,795.666667,26.133333,4.733333,16.366667,1
1,Argentina,ARG,América del Sur,Maíz,2011,3371781,8.45,28485921,164,5557,0.030943,51.0,69.0,14.753333,4.666667,795.666667,26.133333,4.733333,16.366667,0
2,Argentina,ARG,América del Sur,Cebada,2011,1661577,2.77,4596917,105,6514,-0.039779,51.0,69.0,14.753333,4.666667,795.666667,26.133333,4.733333,16.366667,0
3,Argentina,ARG,América del Sur,Té,2011,188518,3.07,577886,236,8527,0.003623,51.0,69.0,14.753333,4.666667,795.666667,26.133333,4.733333,16.366667,0
4,Argentina,ARG,América del Sur,Algodón,2011,1638327,1.18,1929756,276,6925,0.007735,51.0,69.0,14.753333,4.666667,795.666667,26.133333,4.733333,16.366667,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,Kenia,KEN,África,Arroz,2020,2415943,6.83,16508239,70,6500,0.023576,111.0,64.0,11.840000,0.000000,938.000000,35.700000,9.100000,22.000000,0
796,Kenia,KEN,África,Maíz,2020,5214983,7.47,38945261,208,1669,-0.013443,111.0,64.0,11.840000,0.000000,938.000000,35.700000,9.100000,22.000000,0
797,Kenia,KEN,África,Té,2020,262109,2.67,700065,85,6487,-0.026137,111.0,64.0,11.840000,0.000000,938.000000,35.700000,9.100000,22.000000,0
798,Kenia,KEN,África,Girasol,2020,3483346,3.64,12677903,229,4437,0.025317,111.0,64.0,11.840000,0.000000,938.000000,35.700000,9.100000,22.000000,0


In [23]:
df['alto_riesgo'].value_counts()

alto_riesgo
0    603
1    197
Name: count, dtype: int64

Definicion de caracteristica

In [24]:
features_num = ['fertilizantes_kg_ha', 'agua_riego_m3_ha', 'temperatura_maxima', 
                'precipitacion_total', 'humedad_relativa_promedio', 'indice_aridez', 'meses_estres_hidrico']
features_cat = ['cultivo']

In [25]:
X = df[features_num + features_cat]
y = df['alto_riesgo']

Procesamiento

In [26]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_num),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), features_cat)
    ])

Modelos en pipeline

In [27]:
# KNN: Modelo básico de proximidad
# Random Forest: Modelo intermedio, robusto y potente
modelos = {
    'KNN': Pipeline([('pre', preprocessor), ('clf', KNeighborsClassifier(n_neighbors=5))]),
    'RandomForest': Pipeline([('pre', preprocessor), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))])
}

Validación

In [28]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1']

Evaluación

In [29]:
print("--- Evaluación del Modelo de Clasificación (Riesgo de Plagas) ---")
for nombre, modelo in modelos.items():
    scores = cross_validate(modelo, X, y, cv=cv, scoring=scoring)
    print(f"\nModelo: {nombre}")
    print(f"  Accuracy (Exactitud):    {scores['test_accuracy'].mean():.4f}")
    print(f"  Precision (Precisión):   {scores['test_precision'].mean():.4f}")
    print(f"  Recall (Sensibilidad):   {scores['test_recall'].mean():.4f}")
    print(f"  F1-Score:                {scores['test_f1'].mean():.4f}")

--- Evaluación del Modelo de Clasificación (Riesgo de Plagas) ---

Modelo: KNN
  Accuracy (Exactitud):    0.8012
  Precision (Precisión):   0.6698
  Recall (Sensibilidad):   0.4004
  F1-Score:                0.4934

Modelo: RandomForest
  Accuracy (Exactitud):    0.8812
  Precision (Precisión):   0.8374
  Recall (Sensibilidad):   0.6442
  F1-Score:                0.7254
